<img src="https://raw.githubusercontent.com/tancao1128/ams261-labs/main/assets/lab20-banner.png" alt="AMS 261 Lab 20 · Summit & Saddle: level curves of f = xy(x − 2)(y + 3) with four saddle points and one summit" width="100%">

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tancao1128/ams261-labs/blob/main/labs/AMS261-Lecture20.ipynb)
[![Launch Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/tancao1128/ams261-labs/main?labpath=labs%2FAMS261-Lecture20.ipynb)
· [Download this notebook (.ipynb)](https://raw.githubusercontent.com/tancao1128/ams261-labs/main/labs/AMS261-Lecture20.ipynb)

## 0 · Mission briefing

**Hook.** You compute $\nabla f(a,b)=\mathbf 0$. So the tangent plane at $(a,b)$ is horizontal.
A classmate says: "Great, that is a peak or a valley." Does a flat tangent plane mean you are at the top?

In this lab we test the claim on one terrain. It is Example 5 of the Lecture 20 page:

$$
f(x,y)=xy(x-2)(y+3).
$$

The page's own predict example, $y^2-x^2$ (Example 3), is the simplest saddle. We meet it again in Section 3.

- **Time:** about 30 minutes.
- **When:** after class, before Lecture 21.
- **Before you start:** read Examples 1, 3, 5, 7 and 8 on the Lecture 20 page.
- **Section numbers.** Sections are numbered 0–13. "(lecture §n)" points to section n of the Lecture 20 page.

**Lab objectives.** By the end of this lab you can

1. find every critical point of a function of two variables with sympy;
2. classify each critical point with the Second Partials Test, and see a saddle in the slices through it;
3. find the absolute extrema on a closed rectangle with a candidate table: interior, edges, corners.

**How to work.** Run the cells from top to bottom with Shift+Enter. A cell marked ✏️ asks you to type something first.

> **Not graded. Practice only.**

In [ ]:
# SETUP: run this cell first. It needs no local files.
import base64, json, math
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch, Rectangle
from matplotlib.collections import LineCollection
from matplotlib.colors import ListedColormap
from IPython.display import HTML, Markdown, display

try:
    from ipywidgets import (FloatSlider, Dropdown, Button, Label, HBox, VBox, Layout,
                            HTML as WidgetHTML, interactive_output)
    HAS_WIDGETS = True
except ImportError:
    HAS_WIDGETS = False

x, y = sp.symbols('x y', real=True)

# Colour tokens. Each colour has one meaning in every plot of this lab.
INK = '#1E293B'        # text, curves, critical points before they are classified
MIN_C = '#2563EB'      # relative minimum (f_xx > 0), positive values
MAX_C = '#DC2626'      # relative maximum (f_xx < 0), negative values
ZERO = '#059669'       # the zero level set f = 0, and lines of critical points
SADDLE_C = '#D97706'   # saddle points (highlight)
CONTOUR = '#94A3B8'    # level curves
SLICE = '#7C3AED'      # a slice line and the curve along it

# Console tokens. They are used only for the frame around each plot.
HUD_NAVY = '#0A0E14'   # top and bottom bars
HUD_PANEL = '#0F1A2B'  # the outer frame
CYAN = '#22D3EE'       # frame text and corner brackets
SCREEN = '#F8FAFC'     # the light screen behind every panel

plt.rcParams.update({'figure.dpi': 100, 'font.size': 11})


def show_math(latex):
    # Display one formula.
    display(Markdown(f"$${latex}$$"))


def show_text(text):
    # Display formatted text.
    display(Markdown(text))


def fmt(v, digits=2):
    # Format a number; never print -0.00.
    v = float(v)
    if abs(v) < 0.5 * 10 ** (-digits):
        v = 0.0
    return f"{v:.{digits}f}"


def legend_line(color, label, ls='-', marker=None):
    # A legend entry for a line or a marker.
    return Line2D([0], [0], color=color, ls=ls, lw=2.2 if marker is None else 0,
                  marker=marker, markersize=9, label=label)


def legend_patch(color, label):
    # A legend entry for a shaded region.
    return Patch(facecolor=color, edgecolor='#64748B', label=label)


def as_number(v):
    # Accept a number or a string such as "9/4".
    return float(sp.sympify(v))


def check_answer(given, target, tol=5e-3):
    # Compare an answer with a target. Returns True or False; never raises.
    try:
        if given is None:
            return False
        if isinstance(target, bool):
            return given is target
        if isinstance(target, (list, tuple)):
            return (len(given) == len(target)
                    and all(check_answer(a, b, tol) for a, b in zip(given, target)))
        return math.isclose(as_number(given), float(target), rel_tol=1e-3, abs_tol=tol)
    except Exception:
        return False


def hud_frame(fig, label, footer='SUMMIT & SADDLE · f = xy(x − 2)(y + 3)'):
    # Put a finished figure inside a navy console frame.
    # All panels, labels and legends sit on one light screen, so axis text stays dark on light.
    side, top, foot, gap, pad = 0.30, 0.40, 0.30, 0.10, 0.12   # sizes in inches
    w, h = fig.get_size_inches()
    W, H = w + 2 * (side + pad), h + top + foot + 2 * (gap + pad)
    fig.set_size_inches(W, H)
    fig.patch.set_facecolor(HUD_PANEL)

    # Dark ink on the light screen.
    for ax in fig.axes:
        ax.set_facecolor(SCREEN)
        ax.tick_params(colors=INK, labelcolor=INK)
        ax.title.set_color(INK)
        ax.xaxis.label.set_color(INK)
        ax.yaxis.label.set_color(INK)
        for spine in ax.spines.values():
            spine.set_color('#64748B')
        if ax.get_legend() is not None:
            for t in ax.get_legend().get_texts():
                t.set_color(INK)

    # The screen, and the area inside it where the panels go.
    sx0, sy0, sx1, sy1 = side, foot + gap, W - side, H - top - gap   # in inches
    box = [sx0 + pad, sy0 + pad, sx1 - pad, sy1 - pad]
    for _ in range(3):
        fig.tight_layout(rect=(box[0] / W, box[1] / H, box[2] / W, box[3] / H))
        bb = fig.get_tightbbox(fig.canvas.get_renderer())   # everything drawn so far, in inches
        over = [sx0 + pad - bb.x0, sy0 + pad - bb.y0, bb.x1 - sx1 + pad, bb.y1 - sy1 + pad]
        if max(over) < 0.01:
            break   # labels and legends all fit on the screen
        box = [box[0] + max(over[0], 0), box[1] + max(over[1], 0),
               box[2] - max(over[2], 0), box[3] - max(over[3], 0)]
    fig.add_artist(Rectangle((sx0 / W, sy0 / H), (sx1 - sx0) / W, (sy1 - sy0) / H,
                             transform=fig.transFigure, facecolor=SCREEN,
                             edgecolor='#334155', lw=1, zorder=-10))

    # A faint grid on the screen, about 24 px apart.
    step = 0.24
    segs = [[(a / W, sy0 / H), (a / W, sy1 / H)] for a in np.arange(sx0 + step, sx1, step)]
    segs += [[(sx0 / W, b / H), (sx1 / W, b / H)] for b in np.arange(sy0 + step, sy1, step)]
    fig.add_artist(LineCollection(segs, colors='#E2E8F0', linewidths=0.6,
                                  transform=fig.transFigure, zorder=-9))

    # Top bar with the section label, and a small footer bar.
    for y_bar, height in [(H - top, top), (0, foot)]:
        fig.add_artist(Rectangle((0, y_bar / H), 1, height / H, transform=fig.transFigure,
                                 facecolor=HUD_NAVY, edgecolor='none', zorder=-8))
    fig.add_artist(Line2D([0, 1], [(H - top) / H] * 2, transform=fig.transFigure, color=CYAN, lw=1.5))
    fig.add_artist(Line2D([0, 1], [foot / H] * 2, transform=fig.transFigure, color=CYAN, lw=0.8, alpha=0.6))
    mono = dict(family='monospace', transform=fig.transFigure, va='center')
    fig.text(side / W, 1 - top / (2 * H), 'AMS 261 · M20 · ' + label, color=CYAN,
             fontsize=10, fontweight='bold', ha='left', **mono)
    fig.text(1 - side / W, 1 - top / (2 * H), '● TERRAIN LINK · LIVE', color=CYAN,
             fontsize=8, ha='right', **mono)
    fig.text(side / W, foot / (2 * H), footer, color=CONTOUR, fontsize=8, ha='left', **mono)
    fig.text(1 - side / W, foot / (2 * H), 'MISSION 20', color=CONTOUR, fontsize=8, ha='right', **mono)

    # Cyan corner brackets just outside the screen.
    off, arm = 0.07, 0.24
    for cx, cy, dx, dy in [(sx0 - off, sy0 - off, 1, 1), (sx1 + off, sy0 - off, -1, 1),
                           (sx0 - off, sy1 + off, 1, -1), (sx1 + off, sy1 + off, -1, -1)]:
        fig.add_artist(Line2D([(cx + dx * arm) / W, cx / W, cx / W],
                              [cy / H, cy / H, (cy + dy * arm) / H],
                              transform=fig.transFigure, color=CYAN, lw=2))
    return fig


CARD = ('max-width:560px;margin-top:6px;border:1px solid #0F1A2B;border-radius:10px;overflow:hidden;'
        'background:#F8FAFC;color:#1E293B;font-family:system-ui,-apple-system,Segoe UI,sans-serif;')
CARD_HEAD = ('background:#0F1A2B;color:#22D3EE;padding:6px 14px;border-bottom:2px solid #22D3EE;'
             'font-family:Menlo,Consolas,monospace;letter-spacing:2px;font-size:12px;')


def check_card(title, rows, summary):
    # A small result card. Each row is (item, mark, words), for example ('max_value_ex5', '✅', 'correct').
    body = ''.join(
        f'<div style="padding:3px 0;color:#1E293B;">'
        f'<span style="display:inline-block;min-width:330px;padding-right:12px;font-family:Menlo,Consolas,monospace;">{item}</span>'
        f'{mark} {words}</div>' for item, mark, words in rows)
    display(HTML(f'<div style="{CARD}"><div style="{CARD_HEAD}">{title}</div>'
                 f'<div style="padding:8px 14px;color:#1E293B;">{body}'
                 f'<div style="margin-top:6px;font-weight:600;color:#1E293B;">{summary}</div></div></div>'))


def mission_complete_card():
    # Shown once the mission log and the retrieval check are both correct.
    display(HTML(f'<div style="{CARD}"><div style="{CARD_HEAD}">✅ MISSION 20 COMPLETE</div>'
                 '<div style="padding:8px 14px;line-height:1.5;color:#1E293B;">'
                 'Your mission log and your retrieval check are both correct.<br>'
                 'One task is left: the prediction for Lecture 21 in Section 13.</div></div>'))


print('Setup complete. Widgets available:', HAS_WIDGETS)

---
## 1 · PREDICT · Terrain scan (lecture §3)

🔮 **STAGE 1/7 · PREDICT** ▰▱▱▱▱▱▱

Our terrain is $f(x,y)=xy(x-2)(y+3)$. Observe that $f=0$ on four lines: $x=0$, $x=2$, $y=0$ and $y=-3$.
These lines cut the plane into nine cells. In each cell, $f$ keeps one sign.

Run the scan below. It shows only the **sign** of $f$. **Do not compute derivatives yet.** Make three guesses.

1. How many critical points does $f$ have?
2. How many of them are peaks (relative maxima)?
3. How many of them are saddle points?

A rough guess is fine. You will check it in Section 4.

In [ ]:
# The terrain of this lab: Example 5 of the Lecture 20 page.
f5 = x*y*(x - 2)*(y + 3)
f5_np = sp.lambdify((x, y), f5, 'numpy')

fig, ax = plt.subplots(figsize=(8.5, 5.6))
X, Y = np.meshgrid(np.linspace(-1.5, 3.5, 400), np.linspace(-4.5, 1.5, 400))
S = np.sign(f5_np(X, Y))
ax.contourf(X, Y, S, levels=[-1.5, 0, 1.5], colors=['#FEE2E2', '#DBEAFE'])
for v in (0, 2):
    ax.axvline(v, color=ZERO, ls='--', lw=2)
for v in (0, -3):
    ax.axhline(v, color=ZERO, ls='--', lw=2)
for cx in (-0.75, 1, 2.75):
    for cy in (-3.75, -1.5, 0.75):
        sign = '+' if f5_np(cx, cy) > 0 else '−'
        ax.text(cx, cy, sign, ha='center', va='center', fontsize=20, fontweight='bold', color=INK)
ax.set_xlim(-1.5, 3.5)
ax.set_ylim(-4.5, 1.5)
ax.set_aspect('equal')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('Section 1 · Terrain scan: the sign of f(x, y) = xy(x − 2)(y + 3)')
ax.legend(handles=[
    legend_patch('#DBEAFE', 'light blue: f > 0 (above sea level)'),
    legend_patch('#FEE2E2', 'light red: f < 0 (below sea level)'),
    legend_line(ZERO, 'green dashed: the four lines where f = 0', ls='--'),
], loc='upper left', bbox_to_anchor=(1.02, 1.0), fontsize=9, frameon=False)
hud_frame(fig, 'SEC 1 · TERRAIN SCAN')
plt.show()

*Figure description.* The plane is shaded by the sign of $f$. Light blue cells have $f>0$ and light red cells have $f<0$.
A plus or minus sign is printed in each of the nine cells. Four green dashed lines show where $f=0$. They form a rectangle with corners $(0,0)$, $(2,0)$, $(0,-3)$ and $(2,-3)$.

*Think about it.* Inside the rectangle, $f>0$, while $f=0$ on its edges. What must happen somewhere inside? Now look at one corner of the rectangle. How do the signs alternate around it?

In [ ]:
# ✏️ PREDICT. Type your three guesses, then run this cell. Do not compute first.
my_n_critical_guess = None   # number of critical points of f
my_n_peaks_guess = None      # number of relative maxima (peaks)
my_n_saddles_guess = None    # number of saddle points

guesses = [my_n_critical_guess, my_n_peaks_guess, my_n_saddles_guess]
if any(g is None for g in guesses):
    print('Prediction not logged yet. Type three guesses above and run this cell again.')
else:
    print(f'Prediction logged: {my_n_critical_guess} critical points, '
          f'{my_n_peaks_guess} peaks, {my_n_saddles_guess} saddles.')
    print('You will check it in Section 4.')

---
## 2 · EXPLORE · Critical points (lecture §3)

🔭 **STAGE 2/7 · EXPLORE** ▰▰▱▱▱▱▱

**Definition (critical point).** Let $f$ be defined on an open region $\mathcal R$ containing $(x_0,y_0)$.
The point $(x_0,y_0)$ is a **critical point** of $f$ if either

1. $\nabla f(x_0,y_0)=\langle 0,0\rangle$, or
2. $f_x(x_0,y_0)$ or $f_y(x_0,y_0)$ does not exist.

Our $f$ is a polynomial. So both partial derivatives exist everywhere, and only type 1 can occur.
The engine `critical_points` solves $f_x=0$ and $f_y=0$ together with sympy.
Then the cell writes the tangent plane at each critical point.

In [ ]:
def critical_points(f, variables=(x, y)):
    # All solutions of grad f = 0, as a sorted list of points (a, b).
    sols = sp.solve([sp.diff(f, v) for v in variables], list(variables), dict=True)
    pts = [(s[variables[0]], s[variables[1]]) for s in sols
           if all(v in s for v in variables)]          # isolated points only
    return sorted(pts, key=lambda p: (float(p[0]), float(p[1])))


fx5, fy5 = sp.diff(f5, x), sp.diff(f5, y)
show_math(r'f_x = ' + sp.latex(sp.factor(fx5)) + r',\qquad f_y = ' + sp.latex(sp.factor(fy5)))

crit5 = critical_points(f5)
names5 = [f'C{k + 1}' for k in range(len(crit5))]
print(f'Number of critical points: {len(crit5)}')
for name, (a, b) in zip(names5, crit5):
    za = f5.subs({x: a, y: b})
    # Tangent plane: z = f(a,b) + f_x(a,b)(x - a) + f_y(a,b)(y - b).
    plane = za + fx5.subs({x: a, y: b})*(x - a) + fy5.subs({x: a, y: b})*(y - b)
    show_math(name + r' = \left(' + sp.latex(a) + ',\ ' + sp.latex(b) + r'\right):\quad '
              + r'\text{tangent plane } z = ' + sp.latex(sp.simplify(plane)))

Every tangent plane printed above is horizontal: $z$ is a constant. So all of them pass the "flat tangent plane" test.
The map below marks them. It does **not** classify them yet.

In [ ]:
def contour_map(ax, xlim=(-1.5, 3.5), ylim=(-4.5, 1.5), mark=True):
    # Level curves of f5, the zero lines, and (optionally) the unclassified critical points.
    X, Y = np.meshgrid(np.linspace(*xlim, 400), np.linspace(*ylim, 400))
    cs = ax.contour(X, Y, f5_np(X, Y), levels=[-6, -3, -1, 0.5, 1, 1.5, 2, 3, 6],
                    colors=CONTOUR, linewidths=1, linestyles='solid')
    ax.clabel(cs, fmt='%g', fontsize=8)
    for v in (0, 2):
        ax.axvline(v, color=ZERO, ls='--', lw=1.6)
    for v in (0, -3):
        ax.axhline(v, color=ZERO, ls='--', lw=1.6)
    if mark:
        for name, (a, b) in zip(names5, crit5):
            ax.plot([float(a)], [float(b)], 'o', color=INK, ms=8)
            ax.annotate(name, (float(a), float(b)), xytext=(7, 6), textcoords='offset points', color=INK)
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_aspect('equal')
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.grid(alpha=0.2)


map_legend = [
    legend_line(CONTOUR, 'grey: level curves f = c, labelled by c'),
    legend_line(ZERO, 'green dashed: the zero level set f = 0', ls='--'),
    legend_line(INK, 'black dots: critical points C1 to C5 (not yet classified)', marker='o'),
]

fig, ax = plt.subplots(figsize=(8.5, 5.6))
contour_map(ax)
ax.set_title('Section 2 · Five points with a horizontal tangent plane')
ax.legend(handles=map_legend, loc='upper left', bbox_to_anchor=(1.02, 1.0), fontsize=9, frameon=False)
hud_frame(fig, 'SEC 2 · CRITICAL POINTS')
plt.show()

*Figure description.* A contour map of $f$ with equal scales on both axes. Grey curves are level curves, labelled by their value.
Green dashed lines are the zero level set. Five black dots, named C1 to C5, are the critical points. Four of them sit at the corners of the green rectangle. One sits at its centre.
Around the centre, the level curves are closed loops. At each corner, the zero lines cross at right angles, like a plus sign.

---
## 3 · EXPLORE · Slice probe (lecture §4)

🔭 **STAGE 2/7 · EXPLORE** ▰▰▱▱▱▱▱

Example 3 of the page probed $y^2-x^2$ along the two axes. The height went up along one axis and down along the other.
We now do the same probe at every critical point of our terrain, in **every** direction.

Fix a critical point $(a,b)$ and a heading $\phi$. The slice through $(a,b)$ is

$$
g(t)=f(a+t\cos\phi,\ b+t\sin\phi),\qquad -0.8\le t\le 0.8 .
$$

Note that $g'(0)=0$, since $\nabla f(a,b)=\mathbf 0$. So the sign of $g''(0)$ tells us how the slice bends at $t=0$.
The readout prints $g''(0)=f_{xx}\cos^2\phi+2f_{xy}\cos\phi\sin\phi+f_{yy}\sin^2\phi$, evaluated at $(a,b)$.

**Your task.** Choose each point C1 to C5 in turn. Move $\phi$ from $0^\circ$ to $180^\circ$.

- At which points do some slices bend **up** while others bend **down**?
- At which point does **every** slice bend down?

If the slider does not appear, use the static cell after it. Change `point_static` and `phi_static` there and run it again.

In [ ]:
def second_partials(f, point):
    # f_xx, f_yy, f_xy and D = f_xx f_yy - f_xy^2 at a point.
    sub = {x: point[0], y: point[1]}
    fxx = sp.diff(f, x, 2).subs(sub)
    fyy = sp.diff(f, y, 2).subs(sub)
    fxy = sp.diff(f, x, y).subs(sub)
    return fxx, fyy, fxy, sp.simplify(fxx*fyy - fxy**2)


def slice_readings(k, phi_deg):
    # The slice g(t) through critical point k at heading phi, and g''(0).
    a, b = float(crit5[k][0]), float(crit5[k][1])
    c, s = math.cos(math.radians(phi_deg)), math.sin(math.radians(phi_deg))
    fxx, fyy, fxy, _ = second_partials(f5, crit5[k])
    g2 = float(fxx)*c*c + 2*float(fxy)*c*s + float(fyy)*s*s
    t = np.linspace(-0.8, 0.8, 401)
    return a, b, c, s, t, f5_np(a + t*c, b + t*s), g2


def bend_words(g2):
    if g2 > 1e-9:
        return 'bends up'
    if g2 < -1e-9:
        return 'bends down'
    return 'flat to second order'


def plot_slice(k=0, phi_deg=0.0):
    a, b, c, s, t, g, g2 = slice_readings(k, phi_deg)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12.5, 4.8),
                                   gridspec_kw={'width_ratios': [1, 1.1]})

    # Left panel: where the slice runs on the map.
    contour_map(ax1)
    ax1.plot(a + t*c, b + t*s, color=SLICE, lw=3)
    ax1.plot([a], [b], 'o', ms=14, mfc='none', mec=INK, mew=2)
    ax1.set_title(f'Slice through {names5[k]} at heading φ = {phi_deg:.0f}°')

    # Right panel: the height along the slice.
    ax2.plot(t, g, color=SLICE, lw=2.4)
    ax2.axhline(float(f5.subs({x: crit5[k][0], y: crit5[k][1]})), color=CONTOUR, ls='--', lw=1.2)
    ax2.plot([0], [g[len(t) // 2]], 'o', color=INK, ms=8)
    ax2.set_xlabel('t (distance along the slice)')
    ax2.set_ylabel('g(t) = height of the terrain')
    ax2.set_title(f"g''(0) = {fmt(g2)}  ·  the slice {bend_words(g2)}")
    ax2.grid(alpha=0.3)
    ax2.legend(handles=[
        legend_line(SLICE, 'purple: the slice line (left) and its height g(t) (right)'),
        legend_line(CONTOUR, 'grey dashed: the height at the critical point', ls='--'),
        legend_line(INK, 'black dot or ring: the chosen critical point', marker='o'),
        legend_line(ZERO, 'green dashed (left): the zero level set f = 0', ls='--'),
    ], loc='upper left', bbox_to_anchor=(1.02, 1.0), fontsize=9, frameon=False)
    hud_frame(fig, 'SEC 3 · SLICE PROBE')
    display(fig)      # display, then close: works the same with or without the slider
    plt.close(fig)


if HAS_WIDGETS:
    # The probe console: a point selector, a heading slider, two axis presets and a text readout.
    point_menu = Dropdown(options=[(f'{n} = ({a}, {b})', k)
                                   for k, (n, (a, b)) in enumerate(zip(names5, crit5))],
                          value=0, description='point', style={'description_width': 'initial'})
    phi_slider = FloatSlider(min=0, max=180, step=1, value=0, description='φ (deg)',
                             readout_format='.0f', continuous_update=False,
                             layout=Layout(width='100%'),
                             style={'handle_color': CYAN, 'description_width': 'initial'})
    # Only the two axis headings are presets. Which slices bend up or down is for you to find.
    presets = [('along x (0°)', 0.0), ('along y (90°)', 90.0)]

    def preset_button(name, value):
        # One button that moves the slider to a preset heading.
        button = Button(description=name, tooltip='Jump to this heading', layout=Layout(width='auto'))
        button.on_click(lambda _button: setattr(phi_slider, 'value', value))
        return button

    readout = WidgetHTML()

    def update_readout(change=None):
        *_, g2 = slice_readings(point_menu.value, phi_slider.value)
        readout.value = (f'<span style="font-family:Menlo,Consolas,monospace;color:#1E293B;">READOUT · '
                         f'{names5[point_menu.value]} · heading φ = {phi_slider.value:.0f}° · '
                         f"g''(0) = {fmt(g2)} · {bend_words(g2)}</span>")

    point_menu.observe(update_readout, names='value')
    phi_slider.observe(update_readout, names='value')
    update_readout()
    plot_screen = interactive_output(plot_slice, {'k': point_menu, 'phi_deg': phi_slider})
    display(VBox([point_menu, phi_slider,
                  HBox([Label('Presets:')] + [preset_button(n, v) for n, v in presets]),
                  plot_screen, readout]))
else:
    print('ipywidgets is not available here. Use the static cell below instead.')

*Figure description.* Left: the contour map from Section 2. A purple segment is the slice line through the chosen critical point, which is circled in black.
Right: the purple curve is the height $g(t)$ along that slice. The grey dashed line is the height at the critical point. The title prints $g''(0)$ and says whether the slice bends up or down.

In [ ]:
# Static version (no widgets needed). Change the two values and run again.
point_static = 1     # 0 to 4 for C1 to C5
phi_static = 45      # heading in degrees
plot_slice(point_static, phi_static)

*Figure description.* The same two panels as above, drawn once for the point and heading stored in `point_static` and `phi_static`.

*Note.* At the four corners, the slices along $x$ and along $y$ run on a green line where $f=0$. So they are flat. Turn the heading to $45^\circ$ or $135^\circ$ to see the bending.

---
## 4 · EXPLAIN · The Second Partials Test (lecture §5)

🧠 **STAGE 3/7 · EXPLAIN** ▰▰▰▱▱▱▱

**Theorem (Second Partials Test).** Suppose the second partial derivatives of $f$ are continuous near $(a,b)$ and $\nabla f(a,b)=\langle 0,0\rangle$. Define the **discriminant**

$$
D=D(a,b)=f_{xx}(a,b)\,f_{yy}(a,b)-\big(f_{xy}(a,b)\big)^2 .
$$

Then:

1. $D>0$ and $f_{xx}(a,b)>0$ ⇒ $f$ has a **relative minimum** at $(a,b)$.
2. $D>0$ and $f_{xx}(a,b)<0$ ⇒ $f$ has a **relative maximum** at $(a,b)$.
3. $D<0$ ⇒ $(a,b,f(a,b))$ is a **saddle point**.
4. $D=0$ ⇒ the test is **inconclusive**.

The engine `classify` applies this test. The cell below runs it at C1 to C5.

In [ ]:
def classify(fxx, D):
    # The Second Partials Test, case by case.
    if D > 0 and fxx > 0:
        return 'relative minimum'
    if D > 0 and fxx < 0:
        return 'relative maximum'
    if D < 0:
        return 'saddle point'
    return 'inconclusive (D = 0)'


results = []
print(f"{'point':<14}{'f_xx':>7}{'f_yy':>7}{'f_xy':>7}{'D':>7}   {'classification':<20}{'f value':>8}")
for name, p in zip(names5, crit5):
    fxx, fyy, fxy, D = second_partials(f5, p)
    kind = classify(fxx, D)
    val = f5.subs({x: p[0], y: p[1]})
    results.append(dict(name=name, point=p, fxx=fxx, D=D, kind=kind, value=val))
    print(f"{name + ' ' + str(p):<14}{str(fxx):>7}{str(fyy):>7}{str(fxy):>7}{str(D):>7}   {kind:<20}{str(val):>8}")

# The quadratic part (1/2)(A X^2 + 2B XY + C Y^2) at C2 = (0, 0), with X = x - 0 and Y = y - 0.
Xs, Ys = sp.symbols('X Y', real=True)
A2, C2_, B2, _ = second_partials(f5, (0, 0))
quad_C2 = sp.expand(sp.Rational(1, 2)*(A2*Xs**2 + 2*B2*Xs*Ys + C2_*Ys**2))
show_math(r'\text{Quadratic part at C2} = \tfrac12\big(AX^2+2BXY+CY^2\big) = ' + sp.latex(quad_C2))

### 4.1 · Hessian-sign map

The discriminant $D(x,y)=f_{xx}f_{yy}-f_{xy}^2$ can be computed at **every** point, not only at critical points.
The map below shades the plane by its sign pattern. Observe that a colour describes how the surface bends there.
It classifies a point only where $\nabla f=\mathbf 0$.

In [ ]:
fxx_np = sp.lambdify((x, y), sp.diff(f5, x, 2), 'numpy')
D_expr = sp.expand(sp.diff(f5, x, 2)*sp.diff(f5, y, 2) - sp.diff(f5, x, y)**2)
D_np = sp.lambdify((x, y), D_expr, 'numpy')
show_math(r'D(x,y) = ' + sp.latex(sp.factor(D_expr)))

X, Y = np.meshgrid(np.linspace(-1.5, 3.5, 500), np.linspace(-4.5, 1.5, 500))
Dv, Fxx = D_np(X, Y), fxx_np(X, Y)
category = np.where(Dv < 0, 0, np.where(Fxx > 0, 1, 2))   # 0 saddle-type, 1 bowl, 2 dome

fig, ax = plt.subplots(figsize=(8.5, 5.6))
ax.contourf(X, Y, category, levels=[-0.5, 0.5, 1.5, 2.5],
            cmap=ListedColormap(['#FEF3C7', '#DBEAFE', '#FEE2E2']))
contour_map(ax, mark=False)
markers = {'saddle point': ('X', SADDLE_C), 'relative maximum': ('^', MAX_C),
           'relative minimum': ('v', MIN_C)}
for r in results:
    m, col = markers.get(r['kind'], ('o', INK))
    ax.plot([float(r['point'][0])], [float(r['point'][1])], marker=m, color=col, ms=13,
            mec=INK, mew=1, ls='none')
ax.set_title('Section 4.1 · Hessian-sign map of f = xy(x − 2)(y + 3)')
ax.legend(handles=[
    legend_patch('#FEF3C7', 'light amber: D < 0 (saddle-shaped bending)'),
    legend_patch('#DBEAFE', 'light blue: D > 0 and f_xx > 0 (bowl-shaped; none here)'),
    legend_patch('#FEE2E2', 'light red: D > 0 and f_xx < 0 (dome-shaped)'),
    legend_line(SADDLE_C, 'amber X: critical point classified as a saddle', marker='X'),
    legend_line(MAX_C, 'red triangle: critical point classified as a relative max', marker='^'),
    legend_line(CONTOUR, 'grey: level curves of f'),
    legend_line(ZERO, 'green dashed: the zero level set f = 0', ls='--'),
], loc='upper left', bbox_to_anchor=(1.02, 1.0), fontsize=9, frameon=False)
hud_frame(fig, 'SEC 4.1 · HESSIAN-SIGN MAP')
plt.show()

*Figure description.* The plane is shaded by the sign of the discriminant. Light amber marks $D<0$. Light red marks $D>0$ with $f_{xx}<0$. Light blue would mark $D>0$ with $f_{xx}>0$, but no light blue region appears. So no critical point in this window can be a relative minimum.
Grey level curves and the green zero lines are drawn on top. The four corners of the green rectangle carry amber X marks: they lie in amber regions, so they are saddles.
The centre carries a red triangle: it lies in a red region, so it is a relative maximum.

<div style="background:#EFF6FF;border-left:6px solid #2563EB;padding:8px 16px;border-radius:8px;color:#1E293B;">

<strong>🧠 Explain.</strong> Near a critical point, write $X=x-a$, $Y=y-b$, $A=f_{xx}$, $B=f_{xy}$, $C=f_{yy}$. The quadratic part of $f$ is $\tfrac12\big(AX^2+2BXY+CY^2\big)$.
The page completes the square, $\tfrac12 A\big(X+\tfrac BA Y\big)^2+\tfrac12\,\tfrac{D}{A}\,Y^2$. This needs $A\neq0$.
At C2 $=(0,0)$ we have $A=C=0$ and $B=-6$. So we use the quadratic part directly: the cell above gives $-6XY$.
<br><br>Explain <b>in two sentences</b>: what is the sign of $-6XY$ along the line $Y=X$, and along the line $Y=-X$? Why does this make C2 a saddle? Connect this to what you saw in the Section 3 probe at C2 with headings $45^\circ$ and $135^\circ$.
<br><br>Then answer the hook: does a flat tangent plane mean you are at the top?

</div>

Write your sentences first. Then run the next cell to compare with your Section 1 prediction.

In [ ]:
# Check your Section 1 prediction against the test.
n_crit = len(results)
n_max = sum(r['kind'] == 'relative maximum' for r in results)
n_min = sum(r['kind'] == 'relative minimum' for r in results)
n_saddle = sum(r['kind'] == 'saddle point' for r in results)
print(f'Critical points: {n_crit}.  Relative maxima: {n_max}.  Relative minima: {n_min}.  Saddles: {n_saddle}.')
print()
if any(g is None for g in guesses):
    print('You did not log a prediction in Section 1. Next time, commit a guess first.')
else:
    for label, guess, true in [('critical points', my_n_critical_guess, n_crit),
                               ('peaks', my_n_peaks_guess, n_max),
                               ('saddles', my_n_saddles_guess, n_saddle)]:
        print(f'{label:<16} your guess: {guess}   test: {true}')

---
## 5 · CHALLENGE · Absolute extrema on a rectangle (lecture §6)

🎯 **STAGE 4/7 · CHALLENGE** ▰▰▰▰▱▱▱

**Theorem (Extreme Value Theorem in 2D).** If $f$ is **continuous** on a **closed and bounded** set $\mathcal D\subset\mathbb R^2$, then $f$ attains its absolute maximum and absolute minimum on $\mathcal D$.

**Page Example 8.** Find the absolute extrema of $f(x,y)=x^2-2xy+2y$ on $\mathcal D=\{(x,y): 0\le x\le 3,\ 0\le y\le 2\}$.

The algorithm has three steps.

1. **Interior.** Find the critical points inside $\mathcal D$.
2. **Boundary.** Reduce $f$ to one variable on each edge. Find its critical points inside the edge. Add the four corners.
3. **Compare.** List every candidate. The largest value is the absolute maximum, and the smallest is the absolute minimum.

**Before you run the cell:** do you expect the absolute maximum inside the rectangle or on its boundary?

In [ ]:
f8 = x**2 - 2*x*y + 2*y
x_lo, x_hi, y_lo, y_hi = 0, 3, 0, 2
candidates = []    # rows (piece, point, value)

# Step 1. Interior critical points.
for (a, b) in critical_points(f8):
    if x_lo < a < x_hi and y_lo < b < y_hi:
        candidates.append(('interior', '—', (a, b), f8.subs({x: a, y: b})))

# Step 2. Each edge, reduced to one variable. Only critical points strictly inside the edge.
t = sp.symbols('t', real=True)
edges = [('bottom $y=0$', (t, y_lo), x_lo, x_hi), ('top $y=2$', (t, y_hi), x_lo, x_hi),
         ('left $x=0$', (x_lo, t), y_lo, y_hi), ('right $x=3$', (x_hi, t), y_lo, y_hi)]
for name, (px, py), lo, hi in edges:
    g_edge = sp.expand(f8.subs({x: px, y: py}, simultaneous=True))
    for t0 in sp.solve(sp.diff(g_edge, t), t):
        if lo < t0 < hi:
            pt = (sp.sympify(px).subs(t, t0), sp.sympify(py).subs(t, t0))
            free = 'x' if name.startswith(('bottom', 'top')) else 'y'
            reduced = sp.factor(g_edge).subs(t, sp.Symbol(free))
            candidates.append((name, '$f=' + sp.latex(reduced) + '$', pt, g_edge.subs(t, t0)))

# The four corners.
for c in [(x_lo, y_lo), (x_hi, y_lo), (x_lo, y_hi), (x_hi, y_hi)]:
    candidates.append(('corner', '—', c, f8.subs({x: c[0], y: c[1]})))

# Step 3. Compare.
table = '| piece | reduced $f$ | point | value of $f$ |\n|---|---|---|---|\n'
for piece, reduced, pt, val in candidates:
    table += f'| {piece} | {reduced} | $({sp.latex(pt[0])},\\ {sp.latex(pt[1])})$ | ${sp.latex(val)}$ |\n'
show_text(table)
values = [val for *_, val in candidates]
ex8_max, ex8_min = max(values), min(values)
ex8_max_points = [pt for _, _, pt, val in candidates if val == ex8_max]
ex8_min_points = [pt for _, _, pt, val in candidates if val == ex8_min]
print()
print(f'Absolute maximum = {ex8_max} at {ex8_max_points}')
print(f'Absolute minimum = {ex8_min} at {ex8_min_points}')

fxx8, fyy8, fxy8, D8 = second_partials(f8, (1, 1))
print(f'\nSecond Partials Test at the interior point (1, 1): D = {D8}, so it is a {classify(fxx8, D8)}.')

### 5.1 · Grid-scan check

A candidate table can miss a point if one step is wrong. So we check it by brute force.
The cell evaluates $f$ on a grid of $301\times 201$ points of $\mathcal D$, spaced $0.01$ apart.
The largest and smallest grid values should agree with the table.

In [ ]:
f8_np = sp.lambdify((x, y), f8, 'numpy')
X8, Y8 = np.meshgrid(np.linspace(x_lo, x_hi, 301), np.linspace(y_lo, y_hi, 201))
Z8 = f8_np(X8, Y8)
i_max, i_min = np.unravel_index(Z8.argmax(), Z8.shape), np.unravel_index(Z8.argmin(), Z8.shape)
print(f'grid max = {fmt(Z8.max(), 4)} at ({fmt(X8[i_max])}, {fmt(Y8[i_max])})')
print(f'grid min = {fmt(Z8.min(), 4)} at ({fmt(X8[i_min])}, {fmt(Y8[i_min])})')
print('Grid agrees with the candidate table:',
      math.isclose(Z8.max(), float(ex8_max), abs_tol=1e-9) and math.isclose(Z8.min(), float(ex8_min), abs_tol=1e-9))

fig, ax = plt.subplots(figsize=(8.5, 7.2))
cs = ax.contour(X8, Y8, Z8, levels=np.arange(0.5, 9, 1), colors=CONTOUR, linewidths=1)
ax.clabel(cs, fmt='%g', fontsize=8)
ax.plot([0, 3, 3, 0, 0], [0, 0, 2, 2, 0], color=INK, lw=2.4)
for *_, pt, val in candidates:
    ax.plot([float(pt[0])], [float(pt[1])], 'o', color=INK, ms=7, zorder=5)
for pt in ex8_max_points:
    ax.plot([float(pt[0])], [float(pt[1])], '^', color=MAX_C, ms=15, mec=INK, zorder=6)
for pt in ex8_min_points:
    ax.plot([float(pt[0])], [float(pt[1])], 'v', color=MIN_C, ms=15, mec=INK, zorder=6)
ax.plot([1], [1], 'X', color=SADDLE_C, ms=14, mec=INK, zorder=6)
ax.set_xlim(-0.3, 3.3)
ax.set_ylim(-0.3, 2.3)
ax.set_aspect('equal')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.grid(alpha=0.2)
ax.set_title('Section 5.1 · Grid-scan check: candidates for the extrema of x² − 2xy + 2y')
ax.legend(handles=[
    legend_line(CONTOUR, 'grey: level curves of f, labelled by value'),
    legend_line(INK, r'thick black: the boundary of the rectangle $\mathcal{D}$'),
    legend_line(INK, 'black dots: every candidate in the table', marker='o'),
    legend_line(MAX_C, 'red triangle: absolute maximum', marker='^'),
    legend_line(MIN_C, 'blue triangle: absolute minimum', marker='v'),
    legend_line(SADDLE_C, 'amber X: the interior critical point (1, 1), a saddle', marker='X'),
], loc='upper center', bbox_to_anchor=(0.5, -0.12), ncol=2, fontsize=9, frameon=False)
hud_frame(fig, 'SEC 5.1 · GRID-SCAN CHECK', footer='EXAMPLE 8 · f = x² − 2xy + 2y on [0, 3] × [0, 2]')
plt.show()

*Figure description.* The thick black rectangle is $\mathcal D$. Grey curves are level curves of $f$, labelled by value. Black dots are all the candidates of the table: four corners, one point on the top edge and one interior point.
The red upward triangle marks the absolute maximum. The two blue downward triangles mark the absolute minimum, which is attained twice. The amber X marks the interior critical point $(1,1)$.

<div style="background:#FFFBEB;border-left:6px solid #D97706;padding:8px 16px;border-radius:8px;color:#1E293B;">

<strong>🎯 Challenge question.</strong> The only interior critical point, $(1,1)$, is a saddle. Explain in two or three sentences why this alone forces <b>both</b> absolute extrema onto the boundary of $\mathcal D$.

</div>

<details>
<summary>Hint (open after you try)</summary>

By the Extreme Value Theorem, both absolute extrema exist. Suppose one of them were at an interior point. Which theorem of lecture §3 says what that point must be? Can a saddle be a relative maximum or a relative minimum?

</details>

---
## 6 · TRANSFER · When the test is silent (lecture §5)

🚀 **STAGE 5/7 · TRANSFER** ▰▰▰▰▰▱▱

<div style="background:#EFF6FF;border-left:6px solid #2563EB;padding:8px 16px;border-radius:8px;color:#1E293B;">

<strong>🚀 Transfer, without the slider.</strong> Page Example 7 has two functions with $D=0$ at their critical points.

1. $f(x,y)=2x^4+y^4$ has the critical point $(0,0)$, and $D=0$ there. Compare $f(x,y)$ with $f(0,0)$ directly. What kind of point is $(0,0)$?
2. $f(x,y)=2-xy^2$ has a whole line of critical points: the $x$-axis. Near $(x_0,0)$, write $f(x,y)-f(x_0,0)$. Decide its sign for $x_0>0$, for $x_0<0$ and for $x_0=0$. Check the <b>full</b> neighbourhood, not one slice.

</div>

In [ ]:
# The engine confirms D = 0. It cannot classify these points.
for f7 in [2*x**4 + y**4, 2 - x*y**2]:
    show_math(r'f = ' + sp.latex(f7) + r':\quad \nabla f = '
              + r'\left\langle ' + sp.latex(sp.diff(f7, x)) + ',\ ' + sp.latex(sp.diff(f7, y)) + r'\right\rangle')
for f7, p in [(2*x**4 + y**4, (0, 0)), (2 - x*y**2, (1, 0)), (2 - x*y**2, (-1, 0)), (2 - x*y**2, (0, 0))]:
    fxx, fyy, fxy, D = second_partials(f7, p)
    print(f'{str(f7):<14} at {str(p):<8}: D = {D} -> {classify(fxx, D)}')

In [ ]:
# ✏️ Optional check for part 2. Type the sign of f(x, y) - 2 near (x0, 0) for x0 > 0: '+', '-' or 'both'.
my_sign_for_positive_x0 = None

if my_sign_for_positive_x0 is None:
    print('Type your answer first.')
else:
    # Sample a small square around (1, 0), away from the line y = 0.
    xs, ys = np.meshgrid(np.linspace(0.9, 1.1, 41), np.linspace(-0.1, 0.1, 41))
    vals = (2 - xs*ys**2) - 2
    seen = '-' if vals.max() <= 0 else ('+' if vals.min() >= 0 else 'both')
    print('Matches the samples near (1, 0):', my_sign_for_positive_x0 == seen)

<details>
<summary>Check your answer</summary>

**Part 1.** No answer is shown here. The Lecture 20 page asks you to do the same kind of comparison yourself, so settle it on paper.

**Part 2.** $f(x,y)-f(x_0,0)=-xy^2$. Close to $(x_0,0)$ with $x_0\neq0$, the sign of $x$ is the sign of $x_0$.
If $x_0>0$, then $-xy^2\le 0$ in that whole neighbourhood, so $(x_0,0)$ is a relative maximum with value $2$.
If $x_0<0$, then $-xy^2\ge 0$, so $(x_0,0)$ is a relative minimum with value $2$.
At $x_0=0$, every neighbourhood contains both signs of $x$. So $-xy^2$ takes both signs, and the origin is the saddle-type point.

</details>

In [ ]:
# Picture check for the transfer (Page Example 7, Part 2). Open "Check your answer" above first.
fig, ax = plt.subplots(figsize=(9, 6))
X7, Y7 = np.meshgrid(np.linspace(-2, 2, 400), np.linspace(-1, 1, 400))
diff7 = -X7*Y7**2                       # f(x, y) - 2 for f = 2 - x y^2
ax.contourf(X7, Y7, np.sign(diff7), levels=[-1.5, -0.5, 0.5, 1.5], colors=['#FEE2E2', '#F8FAFC', '#DBEAFE'])
cs = ax.contour(X7, Y7, 2 - X7*Y7**2, levels=[1.4, 1.7, 1.9, 2.1, 2.3, 2.6], colors=CONTOUR, linewidths=1)
ax.clabel(cs, fmt='%g', fontsize=8)
ax.axhline(0, color=ZERO, lw=3)
ax.plot([0], [0], 'X', color=SADDLE_C, ms=14, mec=INK)
ax.text(1.0, -0.6, 'f < 2 on both sides:\naxis points are relative maxima', ha='center', color=INK, fontsize=10)
ax.text(-1.0, -0.6, 'f > 2 on both sides:\naxis points are relative minima', ha='center', color=INK, fontsize=10)
ax.set_xlim(-2, 2)
ax.set_ylim(-1, 1)
ax.set_aspect('equal')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('Transfer: the sign of f(x, y) − 2 for f = 2 − xy²')
ax.legend(handles=[
    legend_patch('#DBEAFE', 'light blue: f > 2'),
    legend_patch('#FEE2E2', 'light red: f < 2'),
    legend_line(CONTOUR, 'grey: level curves of f, labelled by value'),
    legend_line(ZERO, 'green: the x-axis, a whole line of critical points (f = 2)'),
    legend_line(SADDLE_C, 'amber X: the origin, the saddle-type point', marker='X'),
], loc='upper center', bbox_to_anchor=(0.5, -0.14), ncol=2, fontsize=9, frameon=False)
hud_frame(fig, 'SEC 6 · TRANSFER · f = 2 − xy²', footer='EXAMPLE 7 · D = 0 ALONG THE WHOLE x-AXIS')
plt.show()

*Figure description.* The thick green line is the $x$-axis. Every point on it is a critical point with $f=2$.
For $x>0$, the plane on both sides of the axis is light red, so $f<2$ there. For $x<0$, it is light blue on both sides, so $f>2$ there.
Grey curves are level curves of $f$. The amber X at the origin is the one point where the colour changes. So a full neighbourhood of the origin contains both signs.

---
## 7 · CREATE · Mission log

📝 **STAGE 6/7 · CREATE** ▰▰▰▰▰▰▱

Fill in the log with your own readings from Sections 4 and 5. Fractions may be typed as strings, for example `"7/3"`.
Then write one sentence in your own words.

In [ ]:
# ✏️ CREATE. Fill in your mission log, then run this cell and the next one.
mission_log = {
    'n_saddles_ex5': None,         # number of saddle points of xy(x-2)(y+3)
    'max_value_ex5': None,         # value of f at its relative maximum
    'D_at_max_ex5': None,          # the discriminant D at that relative maximum
    'abs_max_ex8': None,           # absolute maximum value of x^2 - 2xy + 2y on [0,3] x [0,2]
    'abs_min_ex8': None,           # absolute minimum value on the same rectangle
    'n_abs_min_points_ex8': None,  # at how many points is the absolute minimum attained?
    'one_sentence': '',            # why can a flat tangent plane sit at neither a peak nor a valley?
}

In [ ]:
# Check the mission log. The card shows only "correct" or "try again", never the expected values.
r_max = [r for r in results if r['kind'] == 'relative maximum'][0]
targets = {
    'n_saddles_ex5': n_saddle,
    'max_value_ex5': r_max['value'],
    'D_at_max_ex5': r_max['D'],
    'abs_max_ex8': ex8_max,
    'abs_min_ex8': ex8_min,
    'n_abs_min_points_ex8': len(ex8_min_points),
}
checks = [(key, check_answer(mission_log[key], target)) for key, target in targets.items()]
rows = [(key, '✅', 'correct') if ok else (key, '🔁', 'try again') for key, ok in checks]
sentence = mission_log['one_sentence'].strip()
rows.append(('one_sentence', '✅', 'logged') if sentence else ('one_sentence', '✏️', 'write one sentence'))

n_ok = sum(ok for _, ok in checks)
mission_log_ok = n_ok == len(checks) and bool(sentence)
summary = f'{n_ok} of {len(checks)} correct.'
summary += ' Sentence logged.' if sentence else ' Sentence not written yet.'
if mission_log_ok:
    summary += ' Mission log complete. Next: the debrief.'
check_card('MISSION LOG · CHECK', rows, summary)
if mission_log_ok and globals().get('quiz_ok', False):
    mission_complete_card()

---
## 8 · DEBRIEF · ⚠️ Common errors: check yourself against these

✅ **STAGE 7/7 · DEBRIEF** ▰▰▰▰▰▰▰

<div style="background:#FEF2F2;border-left:6px solid #DC2626;padding:8px 16px;border-radius:8px;color:#1E293B;">

<strong>⚠️ Common errors.</strong>

- **Calling every critical point a peak or a valley.** A flat tangent plane is a necessary condition, not a sufficient one. Four of the five critical points in Section 2 are saddles.
- **Stopping at the interior.** On a closed bounded region the absolute max or min is very often on the **boundary**. In Section 5 both of them are.
- **Treating $D=0$ as a saddle.** It means *inconclusive*, not *saddle*. In Section 6, $2-xy^2$ has relative maxima and minima where $D=0$.
- **Checking only one direction when $D=0$.** Check the full neighbourhood. In Section 6, one slice would not show the sign pattern of $-xy^2$.
- **Checking only $f_{xx}$ when $D<0$.** The saddle conclusion does not depend on the sign of $f_{xx}$. At the corners in Section 4, $f_{xx}=0$.
- **Forgetting the corners** of a polygonal region. They are not interior points of any edge, so an edge parametrization alone can skip them.
- **Missing critical points where a partial derivative fails to exist.** Example 2 of the page has one (see Section 12.2).

</div>

## 9 · DEBRIEF · Retrieval check: mini-quiz

✅ **STAGE 7/7 · DEBRIEF** ▰▰▰▰▰▰▰

Fill in `student_answers` from memory, then run the cell. Numbers may be decimals or exact strings such as `"9/4"`.
The check card shows only **correct** or **try again**. After your second try you may set `reveal = True` to see the key.

In [ ]:
# ✏️ Mini-quiz on the lecture's own examples.
student_answers = {
    'D_ex1_at_min': None,            # D at the critical point of 2x^2 + y^2 + 8x - 6y + 20
    'min_value_ex1': None,           # the relative minimum value of that function
    'D_y2_minus_x2': None,           # D at (0, 0) for f = y^2 - x^2
    'n_critical_points_ex5': None,   # number of critical points of xy(x-2)(y+3)
    'D_zero_means_saddle': None,     # True or False
}
reveal = False   # set to True only after your second try

_key = json.loads(base64.b64decode('eyJEX2V4MV9hdF9taW4iOiA4LCAibWluX3ZhbHVlX2V4MSI6IDMsICJEX3kyX21pbnVzX3gyIjogLTQsICJuX2NyaXRpY2FsX3BvaW50c19leDUiOiA1LCAiRF96ZXJvX21lYW5zX3NhZGRsZSI6IGZhbHNlfQ=='))
if any(v is not None for v in student_answers.values()):
    quiz_attempts = globals().get('quiz_attempts', 0) + 1
else:
    quiz_attempts = globals().get('quiz_attempts', 0)

n_correct = 0
rows = []
for name, target in _key.items():
    ok = check_answer(student_answers[name], target)
    n_correct += ok
    rows.append((name, '✅', 'correct') if ok else (name, '🔁', 'try again'))
quiz_ok = n_correct == len(_key)
check_card('RETRIEVAL · CHECK', rows, f'{n_correct} of {len(_key)} correct · attempt {quiz_attempts}')

if reveal and quiz_attempts < 2:
    print('Try at least twice before you reveal the key.')
elif reveal:
    print('Key:', _key)
elif quiz_ok and globals().get('mission_log_ok', False):
    mission_complete_card()
elif quiz_ok:
    print('All correct. Finish the mission log in Stage 6 to complete the mission.')

## 10 · DEBRIEF · 🤖 AI Learning Companion

✅ **STAGE 7/7 · DEBRIEF** ▰▰▰▰▰▰▰

<div style="background:#ECFDF5;border-left:6px solid #059669;padding:6px 14px;border-radius:8px;color:#1E293B;">
<strong>🤖 Student-facing AI contract — paste this at the start of every AI session</strong>
<br>1. Ask me what I have already tried before you explain anything.
<br>2. Do not give a complete solution until I have made a prediction or proposed a next step.
<br>3. Point to the first gap in my reasoning, not every later consequence.
<br>4. Make me verify the result by units, sign, substitution, a graph, or a limiting case.
<br>5. End with one transfer question that changes the context.
</div>

<div style="background:#F8FAFC;border-left:6px solid #94A3B8;padding:6px 14px;border-radius:8px;margin-top:6px;color:#1E293B;">
<strong>🔎 Verification rule.</strong> Treat an AI answer as a conjecture. Check it against the definitions, theorems and figures on this page. Bring any disagreement to class. An AI answer never overrides a checked definition or proof. Neither does an illustration.
</div>

**Prompt for this lab: Error Hunter.** Paste your own critical-point or absolute-extrema solution in place of `[paste my work]`.

```
Here is my solution to a critical-point classification or absolute-extrema problem:
[paste my work]
Find the FIRST invalid step. Do not fix it. Instead:
(1) quote that step, (2) name the underlying misconception in one sentence,
(3) give me a two-line counterexample or sanity check that exposes it,
(4) ask me to repair the step myself.
If my solution is correct, instead give me a harder variant that breaks the
method I used, and ask me why it breaks.
```

## 11 · DEBRIEF · 🔒 Independence check

✅ **STAGE 7/7 · DEBRIEF** ▰▰▰▰▰▰▰

<div style="background:#ECFDF5;border-left:6px solid #059669;padding:8px 16px;border-radius:8px;color:#1E293B;">
<strong>🔒 Independence rule.</strong> <i>No AI, no hints, new numbers. This shows that you can do it alone.</i>
</div>

Let $h(x,y)=x^3-12x+y^2-4y$.

1. Find all critical points of $h$.
2. Classify each one with the Second Partials Test, and give the value of $h$ there.
3. Is the relative minimum also an absolute minimum of $h$ on the whole plane? Explain in one sentence.

Do this on paper, without AI and without the worked examples. It is not checked here. It is practice for HW8.

---
## 12 · OPTIONAL · Review examples from the lecture page

**OPTIONAL · outside the seven stages.** Skip this part if you are short of time. The mission ends with the prediction for Lecture 21 below.

These are Examples 4 and 2 of the Lecture 20 page, solved with the engines. Try each one on paper before you run its cell.

### 12.1 · Example 4 · Redo Examples 1 and 3 with the test

1. $f(x,y)=2x^2+y^2+8x-6y+20$.
2. $f(x,y)=y^2-x^2$.

In [ ]:
for f4 in [2*x**2 + y**2 + 8*x - 6*y + 20, y**2 - x**2]:
    show_math(r'f(x,y) = ' + sp.latex(f4))
    for p in critical_points(f4):
        fxx, fyy, fxy, D = second_partials(f4, p)
        show_math(r'\left(' + sp.latex(p[0]) + ',\ ' + sp.latex(p[1]) + r'\right):\ '
                  + f'f_{{xx}}={fxx},\ f_{{yy}}={fyy},\ f_{{xy}}={fxy},\ D={D}'
                  + r'\ \Rightarrow\ \text{' + classify(fxx, D) + '}'
                  + r',\quad f = ' + sp.latex(f4.subs({x: p[0], y: p[1]})))

### 12.2 · Example 2 · A critical point where the partial derivatives fail to exist

Let $f(x,y)=1-(x^2+y^2)^{1/3}$. The cell computes $f_x(0,0)$ from its limit definition, $\lim_{h\to0}\frac{f(h,0)-f(0,0)}{h}$, from each side.

In [ ]:
h_ = sp.symbols('h', real=True)
f2 = 1 - (x**2 + y**2)**sp.Rational(1, 3)
quotient = sp.simplify((f2.subs({x: h_, y: 0}) - f2.subs({x: 0, y: 0})) / h_)
show_math(r'\frac{f(h,0)-f(0,0)}{h} = ' + sp.latex(quotient))
right, left = sp.limit(quotient, h_, 0, '+'), sp.limit(quotient, h_, 0, '-')
print('limit from the right:', right, '| limit from the left:', left)
print('So f_x(0,0) does not exist, and (0,0) is a critical point of type 2.')
print('Since (x^2 + y^2)^(1/3) >= 0, f(x, y) <= 1 = f(0, 0): a relative (absolute) maximum.')

---
## 13 · DEBRIEF · Prediction for Lecture 21

✅ **STAGE 7/7 · DEBRIEF** ▰▰▰▰▰▰▰

The mission is complete when the Stage 6 log and the retrieval check both show ✅ for every item.
Your last task is one prediction.

<div style="background:#0A0E14;border:1px solid #22D3EE;border-radius:10px;padding:10px 16px;color:#E2E8F0;">
<span style="color:#22D3EE;font-family:Menlo,Consolas,monospace;letter-spacing:1px;font-size:12px;">MISSION 20 · FINAL PROMPT · PREDICTION FOR LECTURE 21</span><br>
Prediction only: do not look it up and do not compute.
<br>Suppose $(x,y)$ must stay on the circle $x^2+y^2=1$. Why might $\nabla f=\mathbf 0$ no longer be the right test for the largest value of $f$?
<br>Write your guess in one sentence. Lecture 21 will test it.
</div>

---
<div style="background:#0F1A2B;border-top:2px solid #22D3EE;color:#94A3B8;padding:8px 16px;font-family:Menlo,Consolas,monospace;font-size:12px;letter-spacing:1px;">
END OF LAB 20 · SUMMIT & SADDLE · © Tan H. Cao, for AMS 261 students
</div>